# Student t model of earthquake magnitude

Copyright 2022 Allen B. Downey

License: [Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [1]:
# If we're running on Colab, install PyMC and ArviZ
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install pymc3
    !pip install arviz

In [2]:
# PyMC generates a FutureWarning we don't need to deal with yet

#import warnings
#warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# %load_ext nb_black

In [4]:
quake_df = pd.read_csv('quake.csv')
quake_df.head()

/tmp/ipykernel_328413/3957669921.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  quake_df = pd.read_csv('quake.csv')


,#YYY,MM,DD,HH,mm,SS.ss,LATITUDE LONGITUDE,Q,MAG,DEPTH,NPH,RMS,EVID,DEPTH NPH
0,1981,1,1,1,49,29.23,33 43.41-118 49.91,A,2.3,0.57,30.0,0.32,3301561.0,NaN
1,1981,1,1,4,13,55.64,33 14.94-115 58.02,A,2.3,1.00,46.0,0.26,3301565.0,NaN
2,1981,1,1,5,20,14.67,34 10.79-117 18.59,A,2.4,4.29,57.0,0.22,3301566.0,NaN
3,1981,1,1,5,39,56.74,34 10.44-117 18.17,C,1.6,4.80,29.0,0.20,3301567.0,NaN
4,1981,1,1,8,23,18.24,33 59.94-117 10.51,A,1.9,14.65,43.0,0.22,3301570.0,NaN


In [5]:
counts = quake_df.groupby('#YYY')['MAG'].count()
counts.describe()

count       42.000000
mean     18841.166667
std      10970.236514
min       5923.000000
25%      12494.750000
50%      15807.000000
75%      20249.750000
max      62822.000000
Name: MAG, dtype: float64

In [6]:
mags = quake_df['MAG']
mags.describe()

count    791329.000000
mean          1.393113
std           0.658813
min           0.000000
25%           0.900000
50%           1.300000
75%           1.800000
max           7.300000
Name: MAG, dtype: float64

In [ ]:
import pymc3 as pm

with pm.Model() as model:
    nu = pm.Exponential('nu', lam=1)
    mu = pm.Normal('mu', mu=0, sigma=10)
    sigma = pm.Exponential('sigma', lam=1)
    mag = pm.StudentT('mag', nu=nu, mu=mu, sigma=sigma, observed=mags.values)
    trace = pm.sample(500)

/tmp/ipykernel_328413/2556867550.py:8: FutureWarning: In v4.0, pm.sample will return an `arviz.InferenceData` object instead of a `MultiTrace` by default. You can pass return_inferencedata=True or return_inferencedata=False to be safe and silence this warning.
  trace = pm.sample(500)
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, mu, nu]


Sampling 4 chains for 1_000 tune and 500 draw iterations (4_000 + 2_000 draws total) took 595 seconds.


In [ ]:
import arviz as az

az.plot_trace(trace)

In [ ]:
nus = trace['nu']
sns.kdeplot(nus)

In [ ]:
mus = trace['mu']
sns.kdeplot(mus)

In [ ]:
sigmas = trace['sigma']
sns.kdeplot(sigmas)

In [ ]:
from scipy.stats import t as t_dist

rvs = t_dist(nus, mus, sigmas).rvs(size=(16000, len(nus)))
rvs.shape

In [ ]:
yearly_max = rvs.max(axis=0)
yearly_max.shape

In [ ]:
from empiricaldist import Surv

surv_yearly_max = Surv.from_seq(yearly_max)
surv_yearly_max.tail()

In [ ]:
surv_yearly_max.plot()

plt.yscale('log')